# Wafer Map of Downstream Status

This notebook loads the saved CSV files and visualizes three downstream states on a wafer map:

- `Pass`
- `Fail`
- `Not tested / no valid downstream record`

The third category combines dies that were never sampled downstream and dies that were sampled but have `test_valid = 0`.

In [ ]:
import sys
sys.path.append('..')

import math

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.utils import load_sources

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 8)

In [ ]:
df_inline, df_downstream = load_sources(input_dir='../data', prefix='synthetic')

print(f"Inline rows: {len(df_inline)}")
print(f"Downstream sampled rows: {len(df_downstream)}")
print(f"Valid downstream rows: {(df_downstream['test_valid'] == 1).sum()}")
print(f"Invalid downstream rows: {(df_downstream['test_valid'] == 0).sum()}")

In [ ]:
df_status = df_inline.merge(
    df_downstream[['wafer_id', 'die_id', 'test_pass', 'test_valid']],
    on=['wafer_id', 'die_id'],
    how='left',
)

def classify_status(row):
    if pd.isna(row['test_valid']) or row['test_valid'] == 0:
        return 'Not tested / no valid downstream record'
    if row['test_pass'] == 1:
        return 'Pass'
    return 'Fail'

df_status['downstream_status'] = df_status.apply(classify_status, axis=1)
df_status['downstream_status'] = pd.Categorical(
    df_status['downstream_status'],
    categories=['Pass', 'Fail', 'Not tested / no valid downstream record'],
    ordered=True,
)

df_status[['wafer_id', 'die_id', 'downstream_status']].head()

In [ ]:
wafer_summary = (
    df_status.groupby(['wafer_id', 'downstream_status'], observed=False)
    .size()
    .unstack(fill_value=0)
    .sort_values(['Fail', 'Not tested / no valid downstream record'], ascending=False)
)

wafer_summary.head()

In [ ]:
overall_counts = df_status['downstream_status'].value_counts().reindex(
    ['Pass', 'Fail', 'Not tested / no valid downstream record']
)

print('Overall downstream status counts:')
print(overall_counts.to_string())
print()
print(f'Number of wafers: {df_status["wafer_id"].nunique()}')

In [ ]:
palette = {
    'Pass': '#2C7FB8',
    'Fail': '#D95F0E',
    'Not tested / no valid downstream record': '#BDBDBD',
}

wafer_ids = sorted(df_status['wafer_id'].unique())
n_cols = 5
n_rows = math.ceil(len(wafer_ids) / n_cols)

x_pad = 4.0
y_pad = 4.0
x_min = df_status['x_mm'].min() - x_pad
x_max = df_status['x_mm'].max() + x_pad
y_min = df_status['y_mm'].min() - y_pad
y_max = df_status['y_mm'].max() + y_pad

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4.4 * n_cols, 4.4 * n_rows),
    sharex=True,
    sharey=True,
)
axes = axes.flatten()

for ax, wafer_id in zip(axes, wafer_ids):
    wafer_df = df_status[df_status['wafer_id'] == wafer_id].copy()

    for status, color in palette.items():
        subset = wafer_df[wafer_df['downstream_status'] == status]
        ax.scatter(
            subset['x_mm'],
            subset['y_mm'],
            s=26,
            marker='s',
            alpha=0.9,
            color=color,
            edgecolors='white',
            linewidths=0.2,
        )

    counts = wafer_df['downstream_status'].value_counts()
    pass_count = int(counts.get('Pass', 0))
    fail_count = int(counts.get('Fail', 0))
    missing_count = int(counts.get('Not tested / no valid downstream record', 0))

    ax.set_title(
        f'{wafer_id}\nP={pass_count}  F={fail_count}  N={missing_count}',
        fontsize=10,
    )
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.grid(True, alpha=0.15)

for ax in axes[len(wafer_ids):]:
    ax.set_visible(False)

for ax in axes[: len(wafer_ids)]:
    ax.set_xlabel('x_mm')
    ax.set_ylabel('y_mm')

legend_handles = [
    plt.Line2D(
        [0], [0],
        marker='s',
        color='w',
        label=label,
        markerfacecolor=color,
        markeredgecolor='white',
        markeredgewidth=0.3,
        markersize=9,
    )
    for label, color in palette.items()
]

fig.legend(
    handles=legend_handles,
    loc='lower center',
    ncol=3,
    frameon=True,
    bbox_to_anchor=(0.5, 0.02),
)
fig.suptitle('Wafer Maps of Downstream Status for All Wafers', fontsize=16, y=0.995)
plt.tight_layout(rect=(0, 0.06, 1, 0.97))
plt.show()